# EDA 8 - Case loactor map after eda 7



### EDA 7 companion figure — mechanism choropleth

Shades ALL London MSOAs by a Frame C mechanism classification that generalises the three case-study narratives, then overlays the seven
case MSOAs (black outline + label) as exemplars of their zones.
 
**Categories (mutually exclusive, from eda4 Frame C typology):**
- genuine cascade   
  - Typ_C_21 == Cascade-led  AND  Casc_Inflow_Share_21 >= 0.5
  - (affluent-inflow-led: the gentrification signature)
- exodus cascade    
  - Typ_C_21 == Cascade-led  AND  Casc_Inflow_Share_21 <  0.5
  - (outflow-driven: the COVID-era false-cascade signature)
- cascade->counter  
  - Typ_C_11 == Cascade-led  AND  Typ_C_21 == Counter-led
- other             
  - everything else (light grey fabric)
 
**Geo Harmonisation**
Camden 024/025 merge: E02000190 inherits the category of E02000189.
 
**Inputs:**  
london_msoa_2011.geojson  (any GeoJSON with MSOA11CD works)
eda4_results_for_phase3_20260626.csv

In [3]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PolyCollection
from matplotlib.lines import Line2D
from shapely.geometry import shape
from shapely.ops import unary_union

plt.rcParams['font.family'] = 'DejaVu Sans'

In [5]:
from pyprojroot import here

ROOT       = here()
sys.path.insert(0, str(ROOT))

DATA_DIR    = ROOT / 'data'
OUT_DIR     = ROOT / 'outputs' / 'case_study'
BOUNDARIES  = DATA_DIR / 'london_msoa_2011.geojson'
EDA4        = ROOT / 'outputs'/'eda4_results_for_phase3_20260626.csv'
OUT_PNG     = OUT_DIR / 'fig_case_locator_choropleth.png'

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
C_IN, C_EX, K_IN = '#c0392b', '#e8a7a0', '#6a51a3'
GREY = '#e9e7e2'

In [7]:
CAT_STYLE = {
    'genuine': dict(fc=C_IN,     alpha=0.95),
    'exodus':  dict(fc=C_EX,     alpha=0.95),
    'flip':    dict(fc=K_IN,     alpha=0.80),
    'other':   dict(fc=GREY,     alpha=1.00),
}

In [8]:
CASES = {
    'E02000191': 'Camden 026',
    'E02000873': 'Tower Hamlets 010',
    'E02000809': 'Southwark 003',
    'E02000561': 'Islington 008',
    'E02000957': 'Wandsworth 035',
    'E02000440': 'Harrow 008',
    'E02000461': 'Harrow 029',
}

In [9]:
LABEL_OFF = {
    'Camden 026':        (-0.105,  0.014),
    'Tower Hamlets 010': ( 0.095,  0.022),
    'Southwark 003':     ( 0.085, -0.048),
    'Islington 008':     ( 0.085,  0.036),
    'Wandsworth 035':    (-0.035, -0.058),
    'Harrow 008':        ( 0.082,  0.030),
    'Harrow 029':        (-0.090, -0.030),
}

In [10]:
INNER_LONDON_LADS = {
    'E09000007','E09000001','E09000011','E09000012','E09000013',
    'E09000014','E09000019','E09000020','E09000022','E09000023',
    'E09000025','E09000028','E09000030','E09000032','E09000033',
}

In [11]:
# ── classify ────────────────────────────────────────────────────────────
e4 = pd.read_csv(EDA4)
 
INFLOW_MIN = 0.25   # sits in the empirical gap (0.16 -> 0.31) among frame-robust cascades
 
def classify(r):
    if r['Typ_C_21'] == 'Cascade-led':
        robust    = r['Typ_A_21'] == 'Cascade-led'          # cascade under London frame too
        inflowled = r['Casc_Inflow_Share_21'] >= INFLOW_MIN # arm not overwhelmingly outflow
        return 'genuine' if (robust and inflowled) else 'exodus'
    if r['Typ_C_11'] == 'Cascade-led' and r['Typ_C_21'] == 'Counter-led':
        return 'flip'
    return 'other'
 
e4['mech'] = e4.apply(classify, axis=1)
cat = dict(zip(e4['msoa11cd'], e4['mech']))
cat['E02000190'] = cat.get('E02000189', 'other')     # Camden 024/025 merge
counts = e4['mech'].value_counts()